# V3 Reward-Function Evaluation

This notebook compares different reward configurations for one fixed RL experiment setting.

What it does:
- loads `grid_search_runs_rewards.csv`
- filters to one fixed algorithm and feature-batch setting
- averages each reward configuration across all tested seeds
- compares reward systems on runtime, runtime score, 2Q gate reduction, and depth reduction
- shows both raw metric plots and normalized comparison views
- optionally saves clean figure files for reports


## 1. Config

Keep the experiment settings here so the rest of the notebook stays simple.


In [ ]:
from pathlib import Path
import re

from IPython.display import display
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="ticks", context="paper")
plt.rcParams["figure.dpi"] = 130

FEATURE_BATCH_ALIASES = {
    "all_features": "plus_all_features",
    "plus_all_no_history": "plus_all_features",
}

# Publication figure colors. Set both background colors to '#FFFFFF' for white figures.
FIGURE_BACKGROUND_COLOR = "#FFFFFF"
AXES_BACKGROUND_COLOR = "#FFFFFF"
FIGURE_TEXT_COLOR = "#24333D"
FIGURE_BORDER_COLOR = "#657681"
FIGURE_GRID_COLOR = "#D4DCE0"
BAR_EDGE_COLOR = "#40515F"

# Raw 2x2 reward comparison: reward_function_*_raw_metrics.
# Main title centered above the complete 2x2 figure.
RAW_MAIN_TITLE_SIZE = 18
# Metric title above each subplot.
RAW_PANEL_TITLE_SIZE = 13
# Reward-configuration names along each subplot y-axis.
RAW_REWARD_NAME_SIZE = 10
# Numbered metric ticks along each subplot x-axis.
RAW_X_TICK_SIZE = 9
# Metric values printed in the right-side value column for each bar.
RAW_BAR_VALUE_SIZE = 10
# Reference-line and reward-emphasis legend beneath the 2x2 figure.
RAW_LEGEND_SIZE = 8.5
# Overall width for the dense 2x2 raw-metrics figure.
RAW_FIGURE_WIDTH = 8
# Height contribution for each reward configuration row in the raw-metrics figure.
RAW_ROW_HEIGHT = 0.54
# Minimum and maximum figure heights for the raw-metrics figure.
RAW_MIN_FIGURE_HEIGHT = 12.0
RAW_MAX_FIGURE_HEIGHT = 22.0
# Thickness of each horizontal bar in the raw-metrics 2x2 figure.
RAW_BAR_HEIGHT = 0.66
# Extra x-axis space reserved for the aligned value labels in each raw-metrics panel.
RAW_VALUE_GUTTER_FRACTION = 0.14

# Normalized reward-function heatmap: reward_function_<split>_heatmap.
# Main heatmap title.
HEATMAP_TITLE_SIZE = 14
# Reward-configuration names along the heatmap y-axis.
HEATMAP_Y_TICK_SIZE = 10
# Metric names along the heatmap x-axis.
HEATMAP_X_TICK_SIZE = 11
# Normalized values printed inside heatmap cells.
HEATMAP_CELL_VALUE_SIZE = 8.5
# Numbered tick values beside the heatmap colorbar.
HEATMAP_COLORBAR_TICK_SIZE = 9
# 'Normalized score' label beside the heatmap colorbar.
HEATMAP_COLORBAR_LABEL_SIZE = 11
# Overall heatmap width and row-height scaling.
HEATMAP_FIGURE_WIDTH = 14.0
HEATMAP_ROW_HEIGHT = 0.40
HEATMAP_MIN_HEIGHT = 8.0
HEATMAP_MAX_HEIGHT = 28.0


def style_evaluation_axes(ax, grid_axis="x"):
    """Apply the shared enclosed, scientific-paper figure style."""
    ax.figure.set_facecolor(FIGURE_BACKGROUND_COLOR)
    ax.set_facecolor(AXES_BACKGROUND_COLOR)
    ax.set_axisbelow(True)
    ax.tick_params(axis="both", colors=FIGURE_TEXT_COLOR, width=0.8)
    ax.xaxis.label.set_color(FIGURE_TEXT_COLOR)
    ax.yaxis.label.set_color(FIGURE_TEXT_COLOR)
    ax.title.set_color(FIGURE_TEXT_COLOR)
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color(FIGURE_BORDER_COLOR)
        spine.set_linewidth(0.8)
    ax.grid(False)
    if grid_axis == "x":
        ax.xaxis.grid(True, color=FIGURE_GRID_COLOR, linewidth=0.65, alpha=0.8)
    elif grid_axis == "y":
        ax.yaxis.grid(True, color=FIGURE_GRID_COLOR, linewidth=0.65, alpha=0.8)


RESULTS_FILE_NAME = "grid_search_runs_rewards.csv"


def resolve_notebook_dir():
    candidates = [
        Path.cwd(),
        Path.cwd() / "machine_learning" / "RL" / "two_opt_chain_rl_v4",
    ]
    for candidate in candidates:
        if not candidate.is_dir():
            continue
        if (candidate / RESULTS_FILE_NAME).exists():
            return candidate
    raise FileNotFoundError(
        "Could not locate the v4 reward-function notebook directory."
    )


NOTEBOOK_DIR = resolve_notebook_dir()
RESULTS_CSV = NOTEBOOK_DIR / RESULTS_FILE_NAME

FILTERS = {
    "algorithm_type": "ppo",
    "feature_batch": "plus_all_features",
}

# Easy switch: choose which split columns are emphasized in summary and plots.
# Options: "validation" or "test"
EVAL_SPLIT = "validation"
if EVAL_SPLIT not in {"validation", "test"}:
    raise ValueError("EVAL_SPLIT must be 'validation' or 'test'.")

ALL_SPLITS = ("validation", "test")

# Optional figure export.
SAVE_FIGURES = True
FIGURE_DIR_NAME = "reward_function_evaluation_figures"

SELECTED_UTILITY_COL = f"{EVAL_SPLIT}_utility"
SELECTED_2Q_COL = f"{EVAL_SPLIT}_avg_two_qubit_gate_reduction_pct"
SELECTED_DEPTH_COL = f"{EVAL_SPLIT}_avg_depth_reduction_pct"
SELECTED_RUNTIME_COL = f"{EVAL_SPLIT}_avg_runtime_seconds"
SELECTED_RUNTIME_SCORE_COL = f"{EVAL_SPLIT}_avg_runtime_score"

CURRENT_SWEEP_COLUMNS = [
    "sweep_id",
    "run_index",
    "total_runs",
    "status",
    "algorithm_name",
    "algorithm_seed",
    "feature_batch",
    "reward_type",
    "runtime_weight",
    "two_qubit_weight",
    "depth_weight",
    "final_reward_weight",
    "total_timesteps",
    "grid_search_timesteps",
    "grid_search_candidates",
    "elapsed_seconds",
    "model_path",
    "best_config",
    "validation_utility",
    "test_utility",
    "validation_avg_two_qubit_gate_reduction_pct",
    "test_avg_two_qubit_gate_reduction_pct",
    "validation_avg_depth_reduction_pct",
    "test_avg_depth_reduction_pct",
    "validation_avg_runtime_seconds",
    "test_avg_runtime_seconds",
    "validation_avg_runtime_score",
    "test_avg_runtime_score",
    "error_message",
]

NUMERIC_COLUMNS = [
    "algorithm_seed",
    "runtime_weight",
    "two_qubit_weight",
    "depth_weight",
    "final_reward_weight",
    "total_timesteps",
    "grid_search_timesteps",
    "grid_search_candidates",
    "elapsed_seconds",
    "validation_utility",
    "test_utility",
    "validation_avg_two_qubit_gate_reduction_pct",
    "test_avg_two_qubit_gate_reduction_pct",
    "validation_avg_depth_reduction_pct",
    "test_avg_depth_reduction_pct",
    "validation_avg_runtime_seconds",
    "test_avg_runtime_seconds",
    "validation_avg_runtime_score",
    "test_avg_runtime_score",
]

FIGURE_DIR = NOTEBOOK_DIR / FIGURE_DIR_NAME
if SAVE_FIGURES:
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Results CSV: {RESULTS_CSV}")
print(f"Filtering to: {FILTERS}")
print(f"Selected split: {EVAL_SPLIT}")


## 2. Load, Filter, And Average Across Seeds

This section keeps only successful runs that match the fixed experiment setting, averages duplicate runs within each `reward_label + seed`, then averages those seed-level results into one summary row per reward system.


In [ ]:
def normalize_feature_batch(value):
    normalized = str(value).strip()
    return FEATURE_BATCH_ALIASES.get(normalized, normalized)


def load_results_log(csv_path):
    if not csv_path.exists():
        return pd.DataFrame()

    frame = pd.read_csv(csv_path)
    if "algorithm_name" in frame.columns and "algorithm_type" not in frame.columns:
        frame = frame.rename(columns={"algorithm_name": "algorithm_type"})

    for column in NUMERIC_COLUMNS:
        if column in frame.columns:
            frame[column] = pd.to_numeric(frame[column], errors="coerce")

    if "algorithm_type" in frame.columns:
        frame["algorithm_type"] = frame["algorithm_type"].astype(str)
    if "algorithm_seed" in frame.columns:
        frame["algorithm_seed"] = pd.to_numeric(frame["algorithm_seed"], errors="coerce").fillna(42).astype(int)
    if "feature_batch" in frame.columns:
        frame["feature_batch"] = frame["feature_batch"].fillna("original").map(normalize_feature_batch)
    return frame


def _fmt_weight(value):
    return f"{float(value):g}"


SHOW_REWARD_NAME = False
SHOW_2Q_WEIGHT = True
SHOW_DEPTH_WEIGHT = True
SHOW_RUNTIME_WEIGHT = True
SHOW_FINAL_WEIGHT = True


def build_reward_label(row):
    reward_type = str(row["reward_type"])
    runtime_weight = _fmt_weight(row["runtime_weight"])
    two_qubit_weight = _fmt_weight(row["two_qubit_weight"])
    depth_weight = _fmt_weight(row["depth_weight"])
    final_reward_weight = _fmt_weight(row["final_reward_weight"])

    if reward_type == "mixed_weighted":
        effective_2q = two_qubit_weight
        effective_depth = depth_weight
        effective_runtime = runtime_weight
        effective_final = final_reward_weight
    elif reward_type == "mixed_equal":
        effective_2q = "0.5"
        effective_depth = "0.5"
        effective_runtime = runtime_weight
        effective_final = final_reward_weight
    elif reward_type == "two_qubit_only":
        effective_2q = "1.0"
        effective_depth = "0.0"
        effective_runtime = runtime_weight
        effective_final = final_reward_weight
    elif reward_type == "depth_only":
        effective_2q = "0.0"
        effective_depth = "1.0"
        effective_runtime = runtime_weight
        effective_final = final_reward_weight
    elif reward_type == "no_runtime":
        effective_2q = "1.0"
        effective_depth = "0.0"
        effective_runtime = "0.0"
        effective_final = final_reward_weight
    elif reward_type == "current_reward":
        effective_2q = "1.0"
        effective_depth = "0.0"
        effective_runtime = runtime_weight
        effective_final = final_reward_weight
    elif reward_type == "lexicographic":
        effective_2q = "1.0"
        effective_depth = depth_weight
        effective_runtime = runtime_weight
        effective_final = final_reward_weight
    else:
        effective_2q = two_qubit_weight
        effective_depth = depth_weight
        effective_runtime = runtime_weight
        effective_final = final_reward_weight

    def fmt_one_decimal(value):
        return f"{float(value):.1f}"

    parts = []
    if SHOW_2Q_WEIGHT:
        parts.append(fmt_one_decimal(effective_2q))
    if SHOW_DEPTH_WEIGHT:
        parts.append(fmt_one_decimal(effective_depth))
    if SHOW_RUNTIME_WEIGHT:
        parts.append(fmt_one_decimal(effective_runtime))
    if SHOW_FINAL_WEIGHT:
        parts.append(fmt_one_decimal(effective_final))

    return "/".join(parts)


all_results = load_results_log(RESULTS_CSV)
if all_results.empty:
    raise ValueError(f"No rows were loaded from {RESULTS_CSV}.")

results = all_results.copy()
if "status" in results.columns:
    results = results[results["status"].astype(str).str.lower() == "success"].copy()

excluded_final_weights = [0.1, 2.0]
results = results[
    ~results["final_reward_weight"].astype(float).apply(
        lambda x: any(np.isclose(x, w) for w in excluded_final_weights)
    )
].copy()
excluded_depth_weights = [0.3]
results = results[
    ~results["depth_weight"].astype(float).apply(
        lambda x: any(np.isclose(x, w) for w in excluded_depth_weights)
    )
].copy()

excluded = (
    np.isclose(results["depth_weight"].astype(float), 1.0)
    & np.isclose(results["two_qubit_weight"].astype(float), 1.0)
    & np.isclose(results["runtime_weight"].astype(float), 0.0)
)
results = results[~excluded].copy()

for key, value in FILTERS.items():
    filter_key = "algorithm_type" if key == "algorithm_name" else key
    filter_value = normalize_feature_batch(value) if filter_key == "feature_batch" else value
    if filter_key not in results.columns:
        raise ValueError(f"Filter column not found in results file: {filter_key}")
    if isinstance(filter_value, float):
        results = results[np.isclose(results[filter_key].astype(float), filter_value)].copy()
    else:
        results = results[results[filter_key] == filter_value].copy()

required_columns = [
    "reward_type",
    "algorithm_seed",
    SELECTED_UTILITY_COL,
    SELECTED_2Q_COL,
    SELECTED_DEPTH_COL,
    SELECTED_RUNTIME_COL,
    SELECTED_RUNTIME_SCORE_COL,
]
missing = [column for column in required_columns if column not in results.columns]
if missing:
    raise ValueError(f"Missing required result columns: {missing}")

results = results.dropna(subset=required_columns).copy()
if results.empty:
    available_feature_batches = sorted(all_results["feature_batch"].dropna().unique().tolist()) if "feature_batch" in all_results.columns else []
    available_algorithms = sorted(all_results["algorithm_type"].dropna().unique().tolist()) if "algorithm_type" in all_results.columns else []
    raise ValueError(
        "No successful runs match the selected algorithm/feature-batch setting. "
        f"Available algorithms: {available_algorithms}. "
        f"Available feature batches: {available_feature_batches}."
    )

results["reward_label"] = results.apply(build_reward_label, axis=1)
results["run_index"] = pd.to_numeric(results["run_index"], errors="coerce")
results["sweep_id"] = results["sweep_id"].fillna("")

metric_columns = [
    "validation_utility",
    "test_utility",
    "validation_avg_two_qubit_gate_reduction_pct",
    "test_avg_two_qubit_gate_reduction_pct",
    "validation_avg_depth_reduction_pct",
    "test_avg_depth_reduction_pct",
    "validation_avg_runtime_seconds",
    "test_avg_runtime_seconds",
    "validation_avg_runtime_score",
    "test_avg_runtime_score",
]

seed_level = (
    results.groupby(["reward_label", "algorithm_seed"], as_index=False)[metric_columns]
    .mean()
)

comparison = (
    seed_level.groupby("reward_label", as_index=False)
    .agg(
        tested_seed_count=("algorithm_seed", "nunique"),
        tested_seeds=("algorithm_seed", lambda values: ", ".join(str(int(v)) for v in sorted(set(values)))),
        validation_utility=("validation_utility", "mean"),
        test_utility=("test_utility", "mean"),
        validation_avg_two_qubit_gate_reduction_pct=("validation_avg_two_qubit_gate_reduction_pct", "mean"),
        test_avg_two_qubit_gate_reduction_pct=("test_avg_two_qubit_gate_reduction_pct", "mean"),
        validation_avg_depth_reduction_pct=("validation_avg_depth_reduction_pct", "mean"),
        test_avg_depth_reduction_pct=("test_avg_depth_reduction_pct", "mean"),
        validation_avg_runtime_seconds=("validation_avg_runtime_seconds", "mean"),
        test_avg_runtime_seconds=("test_avg_runtime_seconds", "mean"),
        validation_avg_runtime_score=("validation_avg_runtime_score", "mean"),
        test_avg_runtime_score=("test_avg_runtime_score", "mean"),
    )
    .sort_values("reward_label")
    .reset_index(drop=True)
)

display(comparison)


## 3. Build A Clean Comparison Table

This section starts from the seed-averaged reward summary and computes ranks plus normalized scores for easier plotting.


In [ ]:
comparison = comparison.copy()
comparison["reward_label"] = comparison["reward_label"].astype(str)

for split_name in ALL_SPLITS:
    comparison[f"{split_name}_2q"] = pd.to_numeric(
        comparison[f"{split_name}_avg_two_qubit_gate_reduction_pct"], errors="coerce"
    )
    comparison[f"{split_name}_depth"] = pd.to_numeric(
        comparison[f"{split_name}_avg_depth_reduction_pct"], errors="coerce"
    )
    comparison[f"{split_name}_runtime"] = pd.to_numeric(
        comparison[f"{split_name}_avg_runtime_seconds"], errors="coerce"
    )
    comparison[f"{split_name}_runtime_score"] = pd.to_numeric(
        comparison[f"{split_name}_avg_runtime_score"], errors="coerce"
    )


def normalize_higher_better(series):
    series = pd.to_numeric(series, errors="coerce")
    min_value = series.min()
    max_value = series.max()
    if pd.isna(min_value) or pd.isna(max_value):
        return pd.Series(np.nan, index=series.index)
    if min_value == max_value:
        return pd.Series(1.0, index=series.index)
    return (series - min_value) / (max_value - min_value)


def normalize_lower_better(series):
    return 1.0 - normalize_higher_better(series)


selected_2q_col = f"{EVAL_SPLIT}_2q"
selected_depth_col = f"{EVAL_SPLIT}_depth"
selected_runtime_col = f"{EVAL_SPLIT}_runtime"
selected_runtime_score_col = f"{EVAL_SPLIT}_runtime_score"
selected_2q_rank_col = f"{EVAL_SPLIT}_2q_rank"
selected_depth_rank_col = f"{EVAL_SPLIT}_depth_rank"
selected_runtime_rank_col = f"{EVAL_SPLIT}_runtime_rank"
selected_runtime_score_rank_col = f"{EVAL_SPLIT}_runtime_score_rank"
selected_2q_score_col = f"{EVAL_SPLIT}_2q_score"
selected_depth_score_col = f"{EVAL_SPLIT}_depth_score"
selected_runtime_norm_col = f"{EVAL_SPLIT}_runtime_score_norm"
selected_runtime_penalty_norm_col = f"{EVAL_SPLIT}_runtime_penalty_score_norm"
selected_composite_score_col = f"{EVAL_SPLIT}_composite_score"

comparison[selected_2q_rank_col] = comparison[selected_2q_col].rank(ascending=False, method="min")
comparison[selected_depth_rank_col] = comparison[selected_depth_col].rank(ascending=False, method="min")
comparison[selected_runtime_rank_col] = comparison[selected_runtime_col].rank(ascending=True, method="min")
comparison[selected_runtime_score_rank_col] = comparison[selected_runtime_score_col].rank(ascending=True, method="min")

comparison[selected_2q_score_col] = normalize_higher_better(comparison[selected_2q_col])
comparison[selected_depth_score_col] = normalize_higher_better(comparison[selected_depth_col])
comparison[selected_runtime_norm_col] = normalize_lower_better(comparison[selected_runtime_col])
comparison[selected_runtime_penalty_norm_col] = normalize_lower_better(comparison[selected_runtime_score_col])
comparison[selected_composite_score_col] = (
    comparison[selected_2q_score_col]
    + comparison[selected_depth_score_col]
    + comparison[selected_runtime_norm_col]
    + comparison[selected_runtime_penalty_norm_col]
) / 4.0

summary_columns = ["reward_label", "tested_seed_count", "tested_seeds"]
for split_name in ALL_SPLITS:
    summary_columns.extend([
        f"{split_name}_2q",
        f"{split_name}_depth",
        f"{split_name}_runtime",
        f"{split_name}_runtime_score",
    ])
summary_columns.extend([
    selected_2q_rank_col,
    selected_depth_rank_col,
    selected_runtime_rank_col,
    selected_runtime_score_rank_col,
    selected_composite_score_col,
])

display(comparison[summary_columns].sort_values(selected_composite_score_col, ascending=False))


## 4. Raw Metric Comparison Plots

These plots compare the seed-averaged reward systems directly on runtime, runtime score, 2Q reduction, and depth reduction for the selected split.

For the 2Q and depth charts, the notebook overlays labeled reference lines for the best single optimizer and the best fixed two-step chain on the matching split.



In [ ]:
# -----------------------------------------------------------------------------
# Raw 2x2 reward-comparison font controls.
# Edit these here to change only this 2x2 figure without searching elsewhere.
# -----------------------------------------------------------------------------
# Main title centered above the full 2x2 figure.
RAW_MAIN_TITLE_SIZE = 20
# Subplot titles, for example '(a) Validation average 2Q reduction (%)'.
RAW_PANEL_TITLE_SIZE = 15
# Reward-setting names along the left side of each subplot.
RAW_REWARD_NAME_SIZE = 15
# Numbered x-axis tick values beneath each subplot.
RAW_X_TICK_SIZE = 12
# Numeric values printed in the right-side value column for each bar.
RAW_BAR_VALUE_SIZE = 15
# Legend text for the reward-emphasis colors and baseline reference lines.
RAW_LEGEND_SIZE = 12
# Small key above the left-side reward labels showing the compact label order.
RAW_LABEL_KEY_SIZE = 15
# Layout controls for the raw-metrics figure.
RAW_LEFT_MARGIN = 0.21
RAW_RIGHT_MARGIN = 0.985
RAW_BOTTOM_MARGIN = 0.08
RAW_TOP_MARGIN = 0.95
RAW_PANEL_HSPACE = 0.2
# Number of reward settings shown in each subplot.
# Each panel keeps only the top N rows ranked by the matching selected-split metric.
RAW_TOP_N_BY_VALIDATION = 10
# Bar color for the top-validation-2Q reduction panel.
RAW_TOP_2Q_COLOR = "#4F8A70"
# Bar color for the top-validation-depth reduction panel.
RAW_TOP_DEPTH_COLOR = "#4F7F8C"
# Slight variant used for runtime of the top-validation-2Q models.
RAW_TOP_2Q_RUNTIME_COLOR = "#7DAA94"
# Slight variant used for runtime of the top-validation-depth models.
RAW_TOP_DEPTH_RUNTIME_COLOR = "#7AA1AB"

# Raw 2x2 panel colors matched to the model evaluator's paper-style plots.
FIGURE_BACKGROUND_COLOR = "#FFFFFF"
AXES_BACKGROUND_COLOR = "#F7F4EA"
FIGURE_TEXT_COLOR = "#24333D"
FIGURE_BORDER_COLOR = "#657681"
FIGURE_GRID_COLOR = "#CDD5D9"
BAR_EDGE_COLOR = "#40515F"

RAW_ROW_BAND_COLOR = "#F3F6F8"
RAW_VALUE_DIVIDER_COLOR = "#B9C3CA"


def classify_reward_emphasis(label):
    """Group reward settings by whether the reward weights favor 2Q, depth, or neither."""
    match = re.search(r"2Q=([-+]?\d*\.?\d+).*?D=([-+]?\d*\.?\d+)", str(label))
    if match is None:
        return "Balanced"
    twoq_weight = float(match.group(1))
    depth_weight = float(match.group(2))
    if np.isclose(twoq_weight, depth_weight):
        return "Balanced"
    return "2Q-focused" if twoq_weight > depth_weight else "Depth-focused"

# Manually edit these so each reference line keeps the same appearance everywhere.
REFERENCE_STYLE_MAP = {
    "2q_single": {"color": "#8F3D38", "linestyle": "--"},
    "2q_chain": {"color": "#8F3D38", "linestyle": "-."},
    "depth_single": {"color": "#4F6B3D", "linestyle": "--"},
    "depth_chain": {"color": "#4F6B3D", "linestyle": "-."},
    "runtime_2q_single": {"color": "#8F3D38", "linestyle": "--"},
    "runtime_2q_chain": {"color": "#8F3D38", "linestyle": "-."},
    "runtime_depth_single": {"color": "#4F6B3D", "linestyle": "--"},
    "runtime_depth_chain": {"color": "#4F6B3D", "linestyle": "-."},
}


def save_figure(fig, stem):
    if not SAVE_FIGURES:
        return
    fig.savefig(
        FIGURE_DIR / f"{stem}.png",
        bbox_inches="tight",
        dpi=300,
        facecolor=fig.get_facecolor(),
    )
    fig.savefig(
        FIGURE_DIR / f"{stem}.pdf",
        bbox_inches="tight",
        facecolor=fig.get_facecolor(),
    )


def format_chain_label(chain_name):
    return str(chain_name).replace("__", " -> ")


def build_best_by_metric_row(frame, rank_column, value_column, style_key, short_label):
    if frame.empty:
        return None

    ranked = (
        frame.groupby("chain_name", as_index=False)
        .agg(
            rank_value=(rank_column, "mean"),
            display_value=(value_column, "mean"),
        )
        .sort_values("rank_value", ascending=False)
    )
    best_row = ranked.iloc[0]
    return {
        "style_key": style_key,
        "label": f"Best {short_label}: {format_chain_label(best_row['chain_name'])}",
        "value": float(best_row["display_value"]),
    }


def load_split_references(split_name):
    split_csv = NOTEBOOK_DIR / "splits" / f"{split_name}.csv"
    split_rows = pd.read_csv(split_csv)

    required_columns = [
        "chain_name",
        "total_two_qubit_gate_reduction_pct",
        "depth_reduction_pct",
        "total_runtime_seconds",
    ]
    missing_columns = [
        column for column in required_columns if column not in split_rows.columns
    ]
    if missing_columns:
        raise ValueError(
            f"Missing required split columns in {split_csv.name}: {missing_columns}"
        )

    split_rows = split_rows.copy()
    split_rows["chain_name"] = split_rows["chain_name"].astype(str)
    split_rows["total_two_qubit_gate_reduction_pct"] = pd.to_numeric(
        split_rows["total_two_qubit_gate_reduction_pct"], errors="coerce"
    )
    split_rows["depth_reduction_pct"] = pd.to_numeric(
        split_rows["depth_reduction_pct"], errors="coerce"
    )
    split_rows["total_runtime_seconds"] = pd.to_numeric(
        split_rows["total_runtime_seconds"], errors="coerce"
    )

    single_rows = split_rows[
        ~split_rows["chain_name"].str.contains("__", regex=False)
    ].copy()
    chain_rows = split_rows[
        split_rows["chain_name"].str.contains("__", regex=False)
    ].copy()

    reference_map = {
        f"{split_name}_2q": [
            build_best_by_metric_row(
                single_rows,
                "total_two_qubit_gate_reduction_pct",
                "total_two_qubit_gate_reduction_pct",
                "2q_single",
                "2Q single",
            ),
            build_best_by_metric_row(
                chain_rows,
                "total_two_qubit_gate_reduction_pct",
                "total_two_qubit_gate_reduction_pct",
                "2q_chain",
                "2Q chain",
            ),
        ],
        f"{split_name}_depth": [
            build_best_by_metric_row(
                single_rows,
                "depth_reduction_pct",
                "depth_reduction_pct",
                "depth_single",
                "depth single",
            ),
            build_best_by_metric_row(
                chain_rows,
                "depth_reduction_pct",
                "depth_reduction_pct",
                "depth_chain",
                "depth chain",
            ),
        ],
        f"{split_name}_runtime_2q": [
            build_best_by_metric_row(
                single_rows,
                "total_two_qubit_gate_reduction_pct",
                "total_runtime_seconds",
                "runtime_2q_single",
                "2Q single runtime",
            ),
            build_best_by_metric_row(
                chain_rows,
                "total_two_qubit_gate_reduction_pct",
                "total_runtime_seconds",
                "runtime_2q_chain",
                "2Q chain runtime",
            ),
        ],
        f"{split_name}_runtime_depth": [
            build_best_by_metric_row(
                single_rows,
                "depth_reduction_pct",
                "total_runtime_seconds",
                "runtime_depth_single",
                "depth single runtime",
            ),
            build_best_by_metric_row(
                chain_rows,
                "depth_reduction_pct",
                "total_runtime_seconds",
                "runtime_depth_chain",
                "depth chain runtime",
            ),
        ],
    }

    return {
        key: [reference for reference in values if reference is not None]
        for key, values in reference_map.items()
    }


METRIC_REFERENCES = {}
for split_name in ALL_SPLITS:
    METRIC_REFERENCES.update(load_split_references(split_name))

plot_columns = ["reward_label"]
for split_name in ALL_SPLITS:
    plot_columns.extend([
        f"{split_name}_runtime",
        f"{split_name}_runtime_score",
        f"{split_name}_2q",
        f"{split_name}_depth",
    ])
plot_frame = comparison[plot_columns].copy()

top_selected_2q = (
    plot_frame
    .sort_values(f"{EVAL_SPLIT}_2q", ascending=False)
    .head(RAW_TOP_N_BY_VALIDATION)
    .copy()
)
top_selected_depth = (
    plot_frame
    .sort_values(f"{EVAL_SPLIT}_depth", ascending=False)
    .head(RAW_TOP_N_BY_VALIDATION)
    .copy()
)

rows_shown = min(RAW_TOP_N_BY_VALIDATION, len(plot_frame))

fig, axes = plt.subplots(
    2,
    1,
    figsize=(RAW_FIGURE_WIDTH, max(RAW_MIN_FIGURE_HEIGHT, min(RAW_MAX_FIGURE_HEIGHT, RAW_ROW_HEIGHT * rows_shown + 4.6))),
    facecolor=FIGURE_BACKGROUND_COLOR,
)
axes = np.atleast_1d(axes).flatten()

plot_specs = [
    (top_selected_2q, (f"{EVAL_SPLIT}_2q",), f"{EVAL_SPLIT}_2q", f"Top 2Q Models: {EVAL_SPLIT.title()} 2Q Reduction (%)", False, "{:.1f}", RAW_TOP_2Q_COLOR),
    (top_selected_depth, (f"{EVAL_SPLIT}_depth",), f"{EVAL_SPLIT}_depth", f"Top Depth Models: {EVAL_SPLIT.title()} Depth Reduction (%)", False, "{:.1f}", RAW_TOP_DEPTH_COLOR),
]

legend_entries = {}

for panel_index, (ax, (selected_frame, reference_keys, column, title, ascending, value_fmt, panel_color)) in enumerate(zip(axes, plot_specs)):
    ordered = selected_frame.sort_values(column, ascending=ascending).copy()
    y_positions = np.arange(len(ordered))
    values = ordered[column].astype(float).to_numpy()

    reference_rows = []
    for reference_key in reference_keys:
        reference_rows.extend(METRIC_REFERENCES.get(reference_key, []))
    reference_values = [float(reference["value"]) for reference in reference_rows]

    data_min = min([0.0, *values.tolist(), *reference_values])
    data_max = max([0.0, *values.tolist(), *reference_values])
    span = data_max - data_min
    if span == 0:
        span = 1.0
    left_limit = data_min - 0.05 * span
    value_divider_x = data_max + 0.045 * span
    value_label_x = data_max + 0.1 * span
    right_limit = data_max + RAW_VALUE_GUTTER_FRACTION * span

    for row_index in range(len(ordered)):
        if row_index % 2 == 0:
            ax.axhspan(row_index - 0.5, row_index + 0.5, color=RAW_ROW_BAND_COLOR, zorder=0)

    bars = ax.barh(
        y_positions,
        values,
        color=panel_color,
        edgecolor=BAR_EDGE_COLOR,
        linewidth=0.6,
        alpha=0.95,
        height=RAW_BAR_HEIGHT,
        zorder=3,
    )
    style_evaluation_axes(ax, grid_axis="x")
    ax.set_xlim(left_limit, right_limit)
    ax.axvline(0, color=FIGURE_BORDER_COLOR, linewidth=0.9, alpha=0.80, zorder=2)
    ax.axvline(value_divider_x, color=RAW_VALUE_DIVIDER_COLOR, linewidth=0.8, alpha=0.95, zorder=2)
    ax.set_yticks(y_positions)
    ax.set_yticklabels(ordered["reward_label"], fontsize=RAW_REWARD_NAME_SIZE)
    ax.invert_yaxis()
    ax.tick_params(axis="x", labelsize=RAW_X_TICK_SIZE)
    ax.tick_params(axis="y", labelsize=RAW_REWARD_NAME_SIZE, length=0, pad=5)
    ax.set_title(
        f"({chr(97 + panel_index)}) {title}",
        fontsize=RAW_PANEL_TITLE_SIZE,
        loc="left",
        pad=8,
        weight="semibold",
    )
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.text(
        -0.21,
        0.95,
        "2Q / D / R / F",
        transform=ax.transAxes,
        ha="left",
        va="bottom",
        fontsize=RAW_LABEL_KEY_SIZE,
        color=FIGURE_TEXT_COLOR,
        weight="bold",
        linespacing=1.0,
    )
    for y, value in zip(y_positions, values):
        ax.text(
            value_label_x,
            y,
            value_fmt.format(value),
            va="center",
            ha="center",
            fontsize=RAW_BAR_VALUE_SIZE,
            color=FIGURE_TEXT_COLOR,
            weight="semibold",
            zorder=4,
        )

    for reference in reference_rows:
        style = REFERENCE_STYLE_MAP[reference["style_key"]]
        line = ax.axvline(
            reference["value"],
            color=style["color"],
            linestyle=style["linestyle"],
            linewidth=1.8,
            alpha=0.90,
            zorder=5,
        )
        legend_entries.setdefault(reference["label"], line)

    for spine in ax.spines.values():
        spine.set_linewidth(0.9)

if legend_entries:
    fig.legend(
        list(legend_entries.values()),
        list(legend_entries.keys()),
        loc="lower center",
        bbox_to_anchor=(0.5, -0.01),
        ncol=2,
        fontsize=RAW_LEGEND_SIZE,
        frameon=True,
        edgecolor=FIGURE_BORDER_COLOR,
    )

fig.suptitle(
    f"Top Reward Configurations by 2Q and Depth Reduction",
    fontsize=RAW_MAIN_TITLE_SIZE,
    y=1.0,
    x=0.55,
    weight="semibold",
    color=FIGURE_TEXT_COLOR,
)
fig.subplots_adjust(
    left=RAW_LEFT_MARGIN,
    right=RAW_RIGHT_MARGIN,
    bottom=RAW_BOTTOM_MARGIN,
    top=RAW_TOP_MARGIN,
    hspace=RAW_PANEL_HSPACE,
)
save_figure(fig, f"reward_function_{EVAL_SPLIT}_raw_metrics")
plt.show()
plt.close(fig)


## 5. Normalized comparison heatmap

This heatmap converts the selected-split metrics into comparable 0-to-1 scores, where higher is always better.


In [ ]:
heatmap_frame = comparison.set_index("reward_label")[[
    selected_2q_score_col,
    selected_depth_score_col,
    selected_runtime_norm_col,
    selected_runtime_penalty_norm_col,
    selected_composite_score_col,
]].rename(columns={
    selected_2q_score_col: "2Q score",
    selected_depth_score_col: "Depth score",
    selected_runtime_norm_col: "Runtime score",
    selected_runtime_penalty_norm_col: "Runtime-penalty score",
    selected_composite_score_col: "Composite score",
})

fig, ax = plt.subplots(
    figsize=(
        HEATMAP_FIGURE_WIDTH,
        max(HEATMAP_MIN_HEIGHT, min(HEATMAP_MAX_HEIGHT, HEATMAP_ROW_HEIGHT * len(heatmap_frame) + 3.0)),
    ),
    facecolor=FIGURE_BACKGROUND_COLOR,
)
style_evaluation_axes(ax, grid_axis=None)
sns.heatmap(
    heatmap_frame,
    annot=True,
    fmt=".3f",
    cmap=sns.light_palette("#355F78", as_cmap=True),
    linewidths=0.5,
    linecolor=AXES_BACKGROUND_COLOR,
    cbar_kws={"label": "Normalized score"},
    annot_kws={"size": HEATMAP_CELL_VALUE_SIZE},
    ax=ax,
)
style_evaluation_axes(ax, grid_axis=None)
ax.set_title(
    f"Normalized Reward-Function Scores ({EVAL_SPLIT.title()}, Seed-Averaged)",
    fontsize=HEATMAP_TITLE_SIZE,
    pad=9,
    weight="semibold",
)
ax.set_xlabel("")
ax.set_ylabel("")
ax.tick_params(axis="x", labelsize=HEATMAP_X_TICK_SIZE, rotation=25)
ax.tick_params(axis="y", labelsize=HEATMAP_Y_TICK_SIZE, rotation=0)
colorbar = ax.collections[0].colorbar
colorbar.ax.tick_params(labelsize=HEATMAP_COLORBAR_TICK_SIZE, colors=FIGURE_TEXT_COLOR)
colorbar.set_label("Normalized score", fontsize=HEATMAP_COLORBAR_LABEL_SIZE, color=FIGURE_TEXT_COLOR)
colorbar.outline.set_edgecolor(FIGURE_BORDER_COLOR)
fig.tight_layout(pad=0.8)
save_figure(fig, f"reward_function_{EVAL_SPLIT}_heatmap")
plt.show()
plt.close(fig)


## 6. Best-By-Category View

This final table highlights which reward system wins each validation category after averaging across the tested seeds.


In [ ]:
winners = pd.DataFrame([
    {
        "category": f"Lowest {EVAL_SPLIT} runtime",
        "reward": comparison.loc[comparison[selected_runtime_col].idxmin(), "reward_label"],
        "value": comparison[selected_runtime_col].min(),
    },
    {
        "category": f"Lowest {EVAL_SPLIT} runtime score",
        "reward": comparison.loc[comparison[selected_runtime_score_col].idxmin(), "reward_label"],
        "value": comparison[selected_runtime_score_col].min(),
    },
    {
        "category": f"Highest {EVAL_SPLIT} 2Q reduction",
        "reward": comparison.loc[comparison[selected_2q_col].idxmax(), "reward_label"],
        "value": comparison[selected_2q_col].max(),
    },
    {
        "category": f"Highest {EVAL_SPLIT} depth reduction",
        "reward": comparison.loc[comparison[selected_depth_col].idxmax(), "reward_label"],
        "value": comparison[selected_depth_col].max(),
    },
    {
        "category": f"Best {EVAL_SPLIT} composite score",
        "reward": comparison.loc[comparison[selected_composite_score_col].idxmax(), "reward_label"],
        "value": comparison[selected_composite_score_col].max(),
    },
])

display(winners)
if SAVE_FIGURES:
    print(f"Saved figures to: {FIGURE_DIR}")


## 7. LaTeX Table Export

This section builds a paper-ready LaTeX table from the seed-averaged reward comparison results for the selected split.


In [ ]:
latex_metric_columns = {
    "2Q Reduction (%)": f"{EVAL_SPLIT}_2q",
    "Depth Reduction (%)": f"{EVAL_SPLIT}_depth",
    "Runtime (s)": f"{EVAL_SPLIT}_runtime",
    "Runtime Score": f"{EVAL_SPLIT}_runtime_score",
}

latex_table_frame = comparison[["reward_label", *latex_metric_columns.values()]].copy()
latex_table_frame = latex_table_frame.rename(columns={
    "reward_label": "Reward Setup",
    **{value: key for key, value in latex_metric_columns.items()},
})
latex_table_frame = latex_table_frame.sort_values("Reward Setup").reset_index(drop=True)

metric_display_columns = [
    "2Q Reduction (%)",
    "Depth Reduction (%)",
    "Runtime (s)",
    "Runtime Score",
]
for column in metric_display_columns:
    latex_table_frame[column] = pd.to_numeric(latex_table_frame[column], errors="coerce")

latex_display_frame = latex_table_frame.copy()
latex_display_frame["2Q Reduction (%)"] = latex_display_frame["2Q Reduction (%)"].map(lambda x: f"{x:.1f}")
latex_display_frame["Depth Reduction (%)"] = latex_display_frame["Depth Reduction (%)"].map(lambda x: f"{x:.1f}")
latex_display_frame["Runtime (s)"] = latex_display_frame["Runtime (s)"].map(lambda x: f"{x:.1f}")
latex_display_frame["Runtime Score"] = latex_display_frame["Runtime Score"].map(lambda x: f"{x:.3f}")

display(latex_display_frame)

latex_caption = f"Seed-averaged {EVAL_SPLIT} performance across reward configurations."
latex_label = f"tab:reward_function_{EVAL_SPLIT}"
latex_column_format = "p{7.2cm}rrrr"

latex_table = latex_display_frame.to_latex(
    index=False,
    escape=True,
    caption=latex_caption,
    label=latex_label,
    column_format=latex_column_format,
)

print(latex_table)
